# Patient-Level IHC Summary Dashboard
Aggregates IHC biomarker results to the patient level and plots polyp type/subtype counts alongside age/gender breakdowns.

**Requires:** `data/ihc_data.xlsx`, `data/ihcData.xlsx`.

In [ ]:
# --- VIM-Polyp setup: resolve DATA_DIR / OUTPUT_DIR (works on Kaggle, Colab, and a local clone) ---
import subprocess
import sys
from pathlib import Path


def _find_or_fetch_src_dir():
    for base in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        candidate = base / "src"
        if (candidate / "vimpolyp_paths.py").exists():
            return candidate
    # Not inside a checkout of the repo (e.g. a bare Colab session) -> clone it.
    repo_dir = Path("VIM-Polyp")
    if not (repo_dir / "src" / "vimpolyp_paths.py").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/biorsd/VIM-Polyp.git", str(repo_dir)],
            check=True,
        )
    return repo_dir / "src"


sys.path.insert(0, str(_find_or_fetch_src_dir()))
from vimpolyp_paths import DATA_DIR, OUTPUT_DIR  # noqa: E402

print(f"Data directory:   {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

# If DATA_DIR is empty, fetch the dataset first, e.g. from the repo root:
#   python scripts/download_zenodo.py --dest data/
#   python scripts/download_kaggle.py --dest data/

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

file_path = str(DATA_DIR / "ihcData.xlsx")
df = pd.read_excel(file_path)
# Find the number of unique patients
unique_patients = df['patient'].nunique()

# Find the unique combinations of patients and polyps
unique_patient_polyp_combinations = df.groupby(['patient', 'polyp']).size().reset_index().rename(columns={0: 'count'})

# Print the results
print("Number of Unique Patients:", unique_patients)
print("Number of Unique Polyps:", df['polyp'].count().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Age Distribution
plt.figure(figsize=(10,6))
sns.histplot(data=df, x='age', bins=30, kde=True)
plt.title('Age Distribution')
plt.show()

mean_age = df['age'].mean()
std_age = df['age'].std()

age_range = f"{mean_age:.2f} +/- {std_age:.2f}"

print("Age Range: ", age_range)

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='gender')
plt.title('Gender Counts')
plt.show()


male_count = df[df['gender'] == 'M'].shape[0]
female_count = df[df['gender'] == 'F'].shape[0]
print("Male Count:", male_count)
print("Female Count:", female_count)

In [ ]:
# Box plot
plt.figure(figsize=(10, 6))
sns.boxplot(x='gender', y='age', data=df)
plt.title('Age Distribution by Gender')
plt.xlabel('Gender')
plt.ylabel('Age')
plt.show()


In [ ]:
df.loc[(df['type'] == 'nonneoplastic') & (df['subtype'] == 'carcinom'), 'type'] = 'carcinom'

In [ ]:
df.loc[df['type'] == 'carcinom', 'type'] = 'carcinoma'

# Bar plot showing the count of polyp types
plt.figure(figsize=(10,6))
sns.countplot(data=df, x='type')
plt.title('Polyp Type Counts')
output_path = str(OUTPUT_DIR / "polyp_type_countsColon.png") 
plt.savefig(output_path, bbox_inches='tight', dpi=300) # bbox_inches='tight' kenar boşluklarını ayarlar, dpi çözünürlüğü artırır.

plt.show()

# Calculate the counts of polyp types
polyp_type_counts = df['type'].value_counts()

# Print the results
print(polyp_type_counts)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
df.loc[df['subtype'] == 'inflamatuar', 'subtype'] = 'inflammatory'
df.loc[df['subtype'] == 'serrated', 'subtype'] = 'hyperplastic'

# Get the distribution of values in the 'subtype' column
subtype_distribution = df['subtype'].value_counts()

# Get the names and counts of the subtypes as separate lists
subtypes = subtype_distribution.index.tolist()
counts = subtype_distribution.values.tolist()

# Create a bar plot
plt.figure(figsize=(10,6))
sns.barplot(x=subtypes, y=counts, alpha=0.8)

plt.title('Distribution of Subtypes')
plt.ylabel('Number of Occurrences', fontsize=12)
plt.xlabel('Subtype', fontsize=12)
output_path = str(OUTPUT_DIR / "polyp_subtype_countsColon.png") 
plt.savefig(output_path, bbox_inches='tight', dpi=300) # bbox_inches='tight' kenar boşluklarını ayarlar, dpi çözünürlüğü artırır.

plt.show()


import pandas as pd

# Perform grouping to count polyp subtypes
polyp_subtype_counts = df.groupby('subtype').size().reset_index(name='count')

# Print the results using DataFrame
print(polyp_subtype_counts)

neoplastic       194
nonneoplastic    116
normal            78
carcinom          16


          subtype  count
0        carcinom     16 - carcinom 
1  hamartomatosis      4 - nonneoplastic
2    hyperplastic     82 - nonneoplastic
3     inflamatuar     21 - nonneoplastic
4          normal     84 - normal
5        serrated      2 - nonneoplastic
6         tubular    154 - neoplastic
7   tubulovillous     31 - neoplastic
8         villous      9 - neoplastic